# User Behavior Analysis
In this notebook, we aggregate the data to understand behavior at the user level (Trip Frequency, Total Spend, Seasonality Preference).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load non-cancelled trips
df = pd.read_csv('../data/non_cancelled_trips.csv')

print(f"Loaded {len(df)} trips")

# --- 1. Calculate Trip Cost per row ---
# Flight Cost
df['flight_cost'] = df['base_fare_usd'].fillna(0)

# Hotel Cost: (price * nights * rooms)
df['hotel_cost'] = (df['hotel_per_room_usd'] * df['nights'] * df['rooms']).fillna(0)

df['total_trip_cost'] = df['flight_cost'] + df['hotel_cost']

# --- 2. Aggregate per User ---
user_agg = df.groupby('user_id').agg(
    trip_count=('session_id', 'count'),
    total_spend=('total_trip_cost', 'sum'),
    avg_spend=('total_trip_cost', 'mean'),
    avg_nights=('nights', 'mean'),
    distinct_airlines=('trip_airline', 'nunique')
).reset_index()

print(f"\nTotal Unique Users in this sample: {len(user_agg)}")
print(user_agg.head())

### Trip Frequency per User

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(user_agg['trip_count'], bins=range(1, 10), discrete=True)
plt.title('Distribution of Trip Count per User')
plt.xlabel('Number of Trips')
plt.ylabel('Number of Users')
plt.show()

print(user_agg['trip_count'].value_counts().sort_index())

### Total Spend per User

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(user_agg['total_spend'], bins=50, kde=True)
plt.title('Distribution of Total Spend per User (USD)')
plt.xlabel('Total Spend (USD)')
plt.show()

print("Descriptive Stats for Total Spend:")
print(user_agg['total_spend'].describe())

### Top Spenders

In [ ]:
top_spenders = user_agg.sort_values(by='total_spend', ascending=False).head(10)
print(top_spenders[['user_id', 'trip_count', 'total_spend', 'avg_spend']])

plt.figure(figsize=(12, 6))
sns.barplot(x='user_id', y='total_spend', data=top_spenders, order=top_spenders['user_id'])
plt.title('Top 10 Users by Total Spend')
plt.show()